In [2]:
import re
import numpy as np
import pandas as pd

In [ ]:

# File paths
data_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.csv"
assembly_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\assembly_details.csv"

# Read CSVs
data_df = pd.read_csv(data_path)
assembly_df = pd.read_csv(assembly_path)

# Merge on LBCode (data) and Code (assembly)
merged_df = data_df.merge(assembly_df[['Code', 'Assembly']], 
                          left_on='LBCode', 
                          right_on='Code', 
                          how='left')

# Drop 'Code' if not needed
merged_df.drop(columns=['Code'], inplace=True)

# Save updated file (with Assembly column)
output_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis_with_Assembly.csv"
merged_df.to_csv(output_path, index=False)

print(f"Merged file saved to: {output_path}")


Merged file saved to: C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis_with_Assembly.csv


In [5]:
data_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.csv"
data_df = pd.read_csv(data_path)


# Create VotePercentage bins (0–10%, 10–20%, …, 90–100%)
bins = range(0, 101, 10)  # [0,10,20,...100]
labels = [f"{i}-{i+10}%" for i in bins[:-1]]  # '0-10%', '10-20%', etc.

data_df['VoteBin'] = pd.cut(
    merged_df['VotePercentage'], 
    bins=bins, 
    labels=labels, 
    include_lowest=True, 
    right=False  # [0–10) instead of (0–10]
)

In [6]:
data_df.head()

,District,LBCode,LBName,WardCode,WardName,Party,Candidate,Gender,Age,Address,...,Rank,Lead,Front,Strength,LBType,Tier,WardTotalVotes,VotePercentage,Assembly,VoteBin
0,ALAPPUZHA,B04031,Thykkattussery,B04031001,Arookkutty,CPI(M),Merijanatt,M,38.0,"Kazhunnukatt, Vaduthala jetty P O, Aroorukkutty",...,2,-82,LDF,-50 to -99,Block,Block,7332,42.03,NaN,40-50%
1,ALAPPUZHA,B04031,Thykkattussery,B04031001,Arookkutty,BJP,Mini,M,47.0,"Kizhakkemaliyekkal, Arookkutty PO",...,3,-2078,NDA,-500 or less,Block,Block,7332,14.81,NaN,10-20%
2,ALAPPUZHA,B04031,Thykkattussery,B04031001,Arookkutty,INC,Animole,M,50.0,"Kavusseri,Arookkutty P O,688535",...,1,82,UDF,50-99,Block,Block,7332,43.15,NaN,40-50%
3,ALAPPUZHA,B04031,Thykkattussery,B04031002,Perumbalam,CPI(M),Sobhanakumari,M,61.0,"Manjuthara,Perumbalam P O",...,1,962,LDF,500+,Block,Block,6901,50.76,NaN,50-60%
4,ALAPPUZHA,B04031,Thykkattussery,B04031002,Perumbalam,INC,Sreerenjini,M,30.0,"Pattekkadu, Perumbalam P O , Cherthala",...,2,-962,UDF,-500 or less,Block,Block,6901,36.82,NaN,30-40%


In [7]:
pd.crosstab(data_df['Rank'], data_df['VoteBin'])


VoteBin,0-10%,10-20%,20-30%,30-40%,40-50%,50-60%,60-70%,70-80%,80-90%,90-100%
Rank,,,,,,,,,,
1,0,1,133,2673,8459,7450,2220,655,217,26
2,73,715,4855,10234,5945,0,0,0,0,0
3,7662,7216,4784,258,0,0,0,0,0,0
4,6422,1131,53,0,0,0,0,0,0,0
5,2395,64,0,0,0,0,0,0,0,0
6,727,3,0,0,0,0,0,0,0,0
7,208,0,0,0,0,0,0,0,0,0
8,55,0,0,0,0,0,0,0,0,0
9,23,0,0,0,0,0,0,0,0,0


In [8]:
iuml_df = data_df[data_df['Party'] == 'IUML']

# Create table of Rank vs VoteBin for IUML
pd.crosstab(iuml_df['Rank'], iuml_df['VoteBin'])


VoteBin,0-10%,10-20%,20-30%,30-40%,40-50%,50-60%,60-70%,70-80%,80-90%,90-100%
Rank,,,,,,,,,,
1,0,0,3,73,466,982,398,144,45,16
2,0,26,125,316,460,0,0,0,0,0
3,26,35,51,4,0,0,0,0,0,0
4,35,16,0,0,0,0,0,0,0,0
5,9,1,0,0,0,0,0,0,0,0
6,3,0,0,0,0,0,0,0,0,0


In [9]:
# Filter for 40–45% and 45–50%
range_40_45 = data_df[(data_df['VotePercentage'] >= 40) & (data_df['VotePercentage'] < 45)]
range_45_50 = data_df[(data_df['VotePercentage'] >= 45) & (data_df['VotePercentage'] < 50)]

# Rank distribution for each range
dist_40_45 = range_40_45['Rank'].value_counts().sort_index()
dist_45_50 = range_45_50['Rank'].value_counts().sort_index()

# Combine into a single table
rank_distribution = pd.DataFrame({
    '40-45%': dist_40_45,
    '45-50%': dist_45_50
}).fillna(0).astype(int)

In [10]:
rank_distribution

,40-45%,45-50%
Rank,,
1,3512,4947
2,4096,1849


In [11]:
# Define bins and labels
bins = [0, 30, 40, 50, 60, 100]   # cut points
labels = ["0-30", "30-40", "40-50", "50-60", "60+"]  # category names

# Create VoteStrength column
data_df['VoteStrength'] = pd.cut(
    data_df['VotePercentage'],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False  # [ ) intervals: 0–30, 30–40, etc.
)

In [13]:
data_df


,District,LBCode,LBName,WardCode,WardName,Party,Candidate,Gender,Age,Address,...,Lead,Front,Strength,LBType,Tier,WardTotalVotes,VotePercentage,Assembly,VoteBin,VoteStrength
0,ALAPPUZHA,B04031,Thykkattussery,B04031001,Arookkutty,CPI(M),Merijanatt,M,38.0,"Kazhunnukatt, Vaduthala jetty P O, Aroorukkutty",...,-82,LDF,-50 to -99,Block,Block,7332,42.03,NaN,40-50%,40-50
1,ALAPPUZHA,B04031,Thykkattussery,B04031001,Arookkutty,BJP,Mini,M,47.0,"Kizhakkemaliyekkal, Arookkutty PO",...,-2078,NDA,-500 or less,Block,Block,7332,14.81,NaN,10-20%,0-30
2,ALAPPUZHA,B04031,Thykkattussery,B04031001,Arookkutty,INC,Animole,M,50.0,"Kavusseri,Arookkutty P O,688535",...,82,UDF,50-99,Block,Block,7332,43.15,NaN,40-50%,40-50
3,ALAPPUZHA,B04031,Thykkattussery,B04031002,Perumbalam,CPI(M),Sobhanakumari,M,61.0,"Manjuthara,Perumbalam P O",...,962,LDF,500+,Block,Block,6901,50.76,NaN,50-60%,50-60
4,ALAPPUZHA,B04031,Thykkattussery,B04031002,Perumbalam,INC,Sreerenjini,M,30.0,"Pattekkadu, Perumbalam P O , Cherthala",...,-962,UDF,-500 or less,Block,Block,6901,36.82,NaN,30-40%,30-40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74688,WAYANAD,M12082,Sulthanbethery,M12082034,Pazhuppathoor,INC,Mercy,M,56.0,Pedikkattukunnel,...,46,UDF,Jan-49,Municipality,Ward,882,44.90,Sulthan Batheri,40-50%,40-50
74689,WAYANAD,M12082,Sulthanbethery,M12082035,Kaivattamoola,JD(S),Rahmath,M,47.0,Illath,...,-210,LDF,-200 to -499,Municipality,Ward,1009,28.54,Sulthan Batheri,20-30%,0-30
74690,WAYANAD,M12082,Sulthanbethery,M12082035,Kaivattamoola,IND,Shaukkathali,F,41.0,Kallikkudathil,...,210,OTH,200-499,Municipality,Ward,1009,49.36,Sulthan Batheri,40-50%,40-50
74691,WAYANAD,M12082,Sulthanbethery,M12082035,Kaivattamoola,BJP,V Sheeja,M,45.0,Kunnumpurath,...,-308,NDA,-200 to -499,Municipality,Ward,1009,18.83,Sulthan Batheri,10-20%,0-30


In [14]:
data_df.to_csv(data_path, index=False)


In [8]:
data_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.csv"
data_df = pd.read_csv(data_path)

# Total votes polled in each assembly
total_votes = (
    data_df.groupby('Assembly')['Votes']
    .sum()
    .reset_index()
    .rename(columns={'Votes': 'TotalVotes'})
)

# Total IUML votes in each assembly
iuml_votes = (
    data_df[data_df['Party'] == 'IUML']
    .groupby('Assembly')['Votes']
    .sum()
    .reset_index()
    .rename(columns={'Votes': 'IUMLVotes'})
)

# Merge the two
assembly_votes = pd.merge(total_votes, iuml_votes, on='Assembly', how='left').fillna(0)

# Add IUML share percentage
assembly_votes['IUMLShare%'] = (assembly_votes['IUMLVotes'] / assembly_votes['TotalVotes'] * 100).round(2)

# Sort by IUMLVotes (descending)
assembly_votes = assembly_votes.sort_values(by='IUMLVotes', ascending=False)

assembly_votes.to_csv('assembly_votes.csv', index=False)    

In [3]:
data_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.csv"
df = pd.read_csv(data_path)

In [5]:
df.describe()

,Age,Votes,Rank,Lead,WardTotalVotes,VotePercentage
count,74666.000000,74693.000000,74693.000000,74693.000000,74693.000000,74666.000000
mean,43.970214,732.541925,2.337046,-432.409476,2635.453470,29.191310
std,10.365065,2227.954433,1.194001,2129.141438,6931.142857,19.028592
min,18.000000,0.000000,1.000000,-43885.000000,0.000000,0.000000
25%,37.000000,126.000000,1.000000,-430.000000,895.000000,11.050000
50%,44.000000,328.000000,2.000000,-171.000000,1084.000000,30.760000
75%,51.000000,533.000000,3.000000,35.000000,1325.000000,44.130000
max,131.000000,48434.000000,12.000000,28983.000000,91329.000000,99.010000


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# --- Input CSV (your path) ---
data_path = r"C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.csv"

# (Optional) backup first
p = Path(data_path)
backup_path = p.with_suffix(".backup.csv")
shutil.copy2(p, backup_path)
print(f"Backup written to: {backup_path}")

# --- Load ---
df = pd.read_csv(data_path, low_memory=False)

# Drop old Strength (if present)
if "Strength" in df.columns:
    df = df.drop(columns=["Strength"])

# Ensure Lead is numeric
df["Lead"] = pd.to_numeric(df.get("Lead", np.nan), errors="coerce")

# --- Compute Strength from Lead ---
STRENGTH_ORDER = [
    "-500 or less", "-200 to -499", "-100 to -199", "-50 to -99", "-1 to -49",
    "0", "1-49", "50-99", "100-199", "200-499", "500+"
]

x = df["Lead"]

conditions = [
    x.le(-500),
    x.gt(-500) & x.le(-200),
    x.gt(-200) & x.le(-100),
    x.gt(-100) & x.le(-50),
    x.gt(-50)  & x.le(-1),
    x.eq(0),
    x.gt(0)    & x.le(49),
    x.ge(50)   & x.le(99),
    x.ge(100)  & x.le(199),
    x.ge(200)  & x.le(499),
    x.ge(500),
]
labels = [
    "-500 or less", "-200 to -499", "-100 to -199", "-50 to -99", "-1 to -49",
    "0", "1-49", "50-99", "100-199", "200-499", "500+"
]

# IMPORTANT: default=None to avoid dtype conflict with string labels
strength = np.select(conditions, labels, default=None)

# Convert to pandas Categorical (None -> NaN automatically)
df["Strength"] = pd.Categorical(strength, categories=STRENGTH_ORDER, ordered=True)

# --- Save back ---
df.to_csv(data_path, index=False, encoding="utf-8-sig")
print(f"Updated file saved to: {data_path}")

# Quick sanity check
print("\nStrength distribution:")
print(pd.value_counts(df["Strength"], dropna=False))


Backup written to: C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.backup.csv
Updated file saved to: C:\Users\OFFICE DESK\OneDrive\Desktop\LSGD_Analysis\data.csv

Strength distribution:
Strength
-200 to -499    20336
-500 or less    14702
-100 to -199     8531
200-499          5640
100-199          5477
-1 to -49        4651
-50 to -99       4612
1-49             4383
50-99            3777
500+             2481
0                 103
Name: count, dtype: int64


C:\Users\OFFICE DESK\AppData\Local\Temp\ipykernel_18468\3855427456.py:63: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(df["Strength"], dropna=False))


In [3]:
# --- Add Assembly to Wards_2025 with PA_2025 (by LBCode) and Corp_2025 overrides (by WardCode) ---

import pandas as pd
import numpy as np

# File paths (edit if needed)
WARDS_CSV = "Wards_2025.csv"
PA_CSV    = "PA_2025.csv"
CORP_CSV  = "Corp_2025.csv"
OUTPUT    = "Wards_2025_with_Assembly.csv"

# LBCodes for which Assembly must be taken from Corp_2025.csv using WardCode (override list)
corp_lb_list = {"C07003", "C13006", "C02002", "C11005", "C01001", "C08004", "M04014"}

# --- Load data (keep codes as strings) ---
wards = pd.read_csv(WARDS_CSV, dtype=str)
pa    = pd.read_csv(PA_CSV,    dtype=str)
corp  = pd.read_csv(CORP_CSV,  dtype=str)

# Ensure key columns exist
required_wards_cols = {"District","LBCode","LBName","WardCode","Type","WardName","Male","Female","Others","TotalVoters"}
missing_wards = required_wards_cols - set(wards.columns)
if missing_wards:
    raise ValueError(f"Wards_2025.csv is missing columns: {missing_wards}")

required_pa_cols = {"District","LBCode","LBName","Assembly"}
missing_pa = required_pa_cols - set(pa.columns)
if missing_pa:
    raise ValueError(f"PA_2025.csv is missing columns: {missing_pa}")

required_corp_cols = {"WardCode","Assembly"}
missing_corp = required_corp_cols - set(corp.columns)
if missing_corp:
    raise ValueError(f"Corp_2025.csv is missing columns: {missing_corp}")

# Normalize key fields (strip spaces)
for df, cols in [(wards, ["LBCode", "WardCode"]),
                 (pa,    ["LBCode"]),
                 (corp,  ["WardCode"])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip()

# De-duplicate lookup tables on their keys to avoid multi-merge explosion
pa_lookup   = pa.loc[:, ["LBCode", "Assembly"]].drop_duplicates(subset=["LBCode"]).rename(columns={"Assembly":"Assembly_PA"})
corp_lookup = corp.loc[:, ["WardCode","Assembly"]].drop_duplicates(subset=["WardCode"]).rename(columns={"Assembly":"Assembly_Corp"})

# --- Merge PA-based Assembly by LBCode ---
merged = wards.merge(pa_lookup, on="LBCode", how="left")

# --- Merge Corp-based Assembly by WardCode (for specific LBCodes only) ---
merged = merged.merge(corp_lookup, on="WardCode", how="left")

# --- Choose final Assembly:
# If LBCode is in corp_lb_list and we have Assembly_Corp, use that; else use Assembly_PA
use_corp = merged["LBCode"].isin(corp_lb_list) & merged["Assembly_Corp"].notna()
merged["Assembly"] = np.where(use_corp, merged["Assembly_Corp"], merged["Assembly_PA"])

# Optional: reorder columns to place Assembly right after LBName (adjust as you prefer)
cols = list(merged.columns)
# Remove helper columns if present
for helper in ["Assembly_PA", "Assembly_Corp"]:
    if helper in cols:
        cols.remove(helper)

# Move "Assembly" to after LBName
def move_after(col_list, col_to_move, after_col):
    if col_to_move in col_list and after_col in col_list:
        col_list.remove(col_to_move)
        idx = col_list.index(after_col) + 1
        col_list.insert(idx, col_to_move)
    return col_list

cols = move_after(cols, "Assembly", "LBName")
merged = merged[cols]

# --- Save the result ---
merged.to_csv(OUTPUT, index=False)

print(f"✅ Done. Wrote: {OUTPUT}")
print(f"Rows: {len(merged):,} | With Assembly filled: {(merged['Assembly'].notna()).sum():,}")


✅ Done. Wrote: Wards_2025_with_Assembly.csv
Rows: 20,998 | With Assembly filled: 20,998
